# Demo: `forecast()` method

This notebook demonstrates the `ForecastingAssistant.forecast()` method for every supported forecaster type.

The `forecast()` method:
1. Profiles the data
2. Generates a plan
3. Generates code
4. Executes the code via `exec()`
5. Returns a `RunResult` with predictions, metrics, and the exact code that was run

**Key guarantee:** `result.code` is the exact script that produced `result.predictions` and `result.metrics`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")
assistant = ForecastingAssistant()

## Synthetic Data

We create datasets for each forecaster type.

In [3]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series with exogenous variable
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)
exog = rng.normal(50, 10, n)

df_single = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": exog,
})

# Multi-series (long format)
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

# Wide format for multivariate
df_wide = pd.DataFrame({
    "date": dates[:n_multi],
    "temperature": 20 + np.cumsum(rng.normal(0, 0.5, n_multi)),
    "humidity": 60 + np.cumsum(rng.normal(0, 0.3, n_multi)),
    "pressure": 1013 + np.cumsum(rng.normal(0, 0.2, n_multi)),
})

print(f"df_single: {df_single.shape}")
print(f"df_multi:  {df_multi.shape}")
print(f"df_wide:   {df_wide.shape}")

df_single: (365, 3)
df_multi:  (600, 3)
df_wide:   (200, 4)


---
## 1. ForecasterRecursive (single series)

The default for single-series forecasting. Uses recursive multi-step prediction.

In [4]:
result_recursive = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
)

print("Plan:", result_recursive.plan.explanation)
print("\nMetrics:")
display(result_recursive.metrics)
print("\nPredictions:")
display(result_recursive.predictions)

Plan: Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 3, 7, 8, 10, 12, 118]. Window features: ['mean(window=7)', 'mean(window=91)']. NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,0.859057,1.118199,0.300112,0.009848



Predictions:


,pred
2022-10-20,85.738240
2022-10-21,85.611453
2022-10-22,88.351413
2022-10-23,93.152032
2022-10-24,94.301703
2022-10-25,91.164140
2022-10-26,86.611982
2022-10-27,84.629980
2022-10-28,83.968107
2022-10-29,87.719652


In [5]:
result_recursive = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    forecaster  = "ForecasterRecursive",
    estimator   = "LGBMRegressor",
)

print("Plan:", result_recursive.plan.explanation)
print("\nMetrics:")
display(result_recursive.metrics)
print("\nPredictions:")
display(result_recursive.predictions)

Plan: Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 3, 7, 8, 10, 12, 118]. Window features: ['mean(window=7)', 'mean(window=91)']. NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,0.859057,1.118199,0.300112,0.009848



Predictions:


,pred
2022-10-20,85.738240
2022-10-21,85.611453
2022-10-22,88.351413
2022-10-23,93.152032
2022-10-24,94.301703
2022-10-25,91.164140
2022-10-26,86.611982
2022-10-27,84.629980
2022-10-28,83.968107
2022-10-29,87.719652


In [6]:
# Inspect the generated code (this is exactly what was executed)
print(result_recursive.code)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2022-10-19'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['promo']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],


In [7]:
# Inspect the generated plan
print(result_recursive.plan)

task_type='single_series' forecaster='ForecasterRecursive' forecaster_kwargs={'lags': [1, 3, 7, 8, 10, 12, 118], 'window_features': [{'stats': ['mean'], 'window_sizes': 7}, {'stats': ['mean'], 'window_sizes': 91}], 'categorical_features': 'auto', 'dropna_from_series': False} estimator='LGBMRegressor' estimator_kwargs={} steps=14 frequency='D' interval=None interval_method=None metric='mean_absolute_error' metrics_to_compute=['mean_absolute_error', 'mean_squared_error', 'mean_absolute_scaled_error', 'mean_absolute_percentage_error'] use_exog=True preprocessing_steps=[] warnings=[] explanation="Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 3, 7, 8, 10, 12, 118]. Window features: ['mean(window=7)', 'mean(window=91)']. NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale."


---
## 2. ForecasterDirect (single series)

Trains one model per step. Better for long horizons or when the DGP changes across horizons.

In [8]:
result_direct = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    forecaster  = "ForecasterDirect",
    estimator   = "Ridge",
)

print("Plan:", result_direct.plan.explanation)
print("\nMetrics:")
display(result_direct.metrics)
print("\nPredictions:")
display(result_direct.predictions)

Plan: Plan: ForecasterDirect + Ridge. Lags: [1, 3, 7, 8, 10, 12, 118]. Window features: ['mean(window=7)', 'mean(window=91)']. NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,0.811934,0.916786,0.283649,0.009322



Predictions:


,pred
2022-10-20,85.248973
2022-10-21,86.039662
2022-10-22,90.014403
2022-10-23,93.771710
2022-10-24,94.746020
2022-10-25,91.758983
2022-10-26,87.074855
2022-10-27,83.945671
2022-10-28,84.664401
2022-10-29,88.540027


---
## 3. ForecasterRecursiveMultiSeries (multi-series)

Global model trained on all series simultaneously. One row per level in metrics.

In [9]:
result_multi = assistant.forecast(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    steps            = 7,
    forecaster       = "ForecasterRecursiveMultiSeries",
    estimator        = "Ridge",
)

print("Plan:", result_multi.plan.explanation)
print("\nMetrics (per series):")
display(result_multi.metrics)
print("\nPredictions:")
display(result_multi.predictions)

Plan: Plan: ForecasterRecursiveMultiSeries + Ridge. Lags: [1, 7, 10, 14, 36, 48, 55]. Window features: ['mean(window=7)', 'mean(window=150)']. NaN rows kept (NaN-tolerant estimator). MASE is scale-independent, enabling fair comparison across differently-scaled series.

Metrics (per series):


,series,MAE,MSE,MASE,MAPE
0,store_A,0.460199,0.336282,0.583835,0.004734
1,store_B,1.964807,5.009403,1.773505,0.009354
2,store_C,2.192522,6.760643,3.272426,0.015189



Predictions:


,level,pred
2022-06-10,store_A,98.163698
2022-06-10,store_B,209.631988
2022-06-10,store_C,146.373017
2022-06-11,store_A,97.945341
2022-06-11,store_B,210.567330
2022-06-11,store_C,146.710525
2022-06-12,store_A,97.519220
2022-06-12,store_B,210.933006
2022-06-12,store_C,146.903255
2022-06-13,store_A,96.802873


---
## 4. ForecasterDirectMultiVariate (multivariate)

Uses all series as features to predict one target series. Direct strategy.

In [10]:
result_multivariate = assistant.forecast(
    data        = df_wide,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
    steps       = 7,
    forecaster  = "ForecasterDirectMultiVariate",
    estimator   = "Ridge",
)

print("Plan:", result_multivariate.plan.explanation)
print("\nMetrics (per series):")
display(result_multivariate.metrics)
print("\nPredictions:")
display(result_multivariate.predictions)

Plan: Plan: ForecasterDirectMultiVariate + Ridge. Lags: [1, 3, 7]. Window features: ['mean(window=7)', 'mean(window=150)']. NaN rows kept (NaN-tolerant estimator). MASE is scale-independent, enabling fair comparison across differently-scaled series.

Metrics (per series):


,series,MAE,MSE,MASE,MAPE
0,temperature,1.093259,1.645405,2.604106,0.042166



Predictions:


,level,pred
2022-06-10,temperature,27.139974
2022-06-11,temperature,27.438645
2022-06-12,temperature,27.320281
2022-06-13,temperature,27.597073
2022-06-14,temperature,27.421161
2022-06-15,temperature,27.344638
2022-06-16,temperature,27.489543


---
## 5. ForecasterStats (ARIMA)

Statistical model. No external estimator needed. Supports native prediction intervals.

In [11]:
result_stats = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    forecaster  = "ForecasterStats",
    interval    = [10, 90],
)

print("Plan:", result_stats.plan.explanation)
print("\nMetrics:")
display(result_stats.metrics)
print("\nPredictions with intervals:")
display(result_stats.predictions)

Plan: Plan: ForecasterStats + Arima. Prediction intervals via native. Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,1.809751,4.291044,0.632236,0.020793



Predictions with intervals:


,pred
2022-10-20,85.864618
2022-10-21,87.063686
2022-10-22,90.684944
2022-10-23,94.059185
2022-10-24,94.781935
2022-10-25,92.196534
2022-10-26,88.461679
2022-10-27,86.311281
2022-10-28,87.223632
2022-10-29,90.547194


---
## 6. ForecasterFoundation (zero-shot)

Pre-trained foundation model (Chronos-2). No training — uses the series context to forecast directly.
Requires `pip install chronos-forecasting` and downloads model weights on first use.

In [12]:
result_foundation = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    forecaster  = "ForecasterFoundation",
    interval    = [10, 90],
)

print("Plan:", result_foundation.plan.explanation)
print("\nMetrics:")
display(result_foundation.metrics)
print("\nPredictions with quantile intervals:")
display(result_foundation.predictions)

Plan: Plan: ForecasterFoundation + Chronos-2. Prediction intervals via native. Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,1.758956,3.746861,0.614491,0.019947



Predictions with quantile intervals:


,level,q_0.1,q_0.5,q_0.9
2022-10-20,sales,84.072388,85.410835,87.089600
2022-10-21,sales,84.254845,85.990181,88.332970
2022-10-22,sales,87.652344,89.744713,91.855461
2022-10-23,sales,91.486290,94.050056,96.422829
2022-10-24,sales,92.512398,95.386223,97.815018
2022-10-25,sales,89.625313,92.535858,95.267807
2022-10-26,sales,84.411621,87.923332,91.069519
2022-10-27,sales,81.769043,85.299797,88.404587
2022-10-28,sales,82.668076,86.405937,89.976288
2022-10-29,sales,86.725792,90.551170,94.013550


---
## 7. ForecasterRecursive with prediction intervals

ML forecaster with bootstrapping-based prediction intervals.

In [13]:
result_intervals = assistant.forecast(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    steps       = 14,
    forecaster  = "ForecasterRecursive",
    estimator   = "Ridge",
    interval    = [10, 90],
)

print("Interval method:", result_intervals.plan.interval_method)
print("\nMetrics:")
display(result_intervals.metrics)
print("\nPredictions with 80% intervals:")
display(result_intervals.intervals)

Interval method: bootstrapping

Metrics:


,series,MAE,MSE,MASE,MAPE
0,sales,1.144499,1.96703,0.39983,0.013135



Predictions with 80% intervals:


,pred,lower_bound,upper_bound
2022-10-20,85.300374,83.943911,86.717888
2022-10-21,85.874471,84.002402,87.445818
2022-10-22,89.685563,87.402005,91.180594
2022-10-23,93.573100,91.757209,95.699978
2022-10-24,94.502138,92.585985,96.685944
2022-10-25,91.687271,89.671068,94.250974
2022-10-26,87.560158,85.121681,90.279294
2022-10-27,84.922192,82.254859,87.286160
2022-10-28,85.707560,83.211438,87.872823
2022-10-29,89.412305,86.602862,92.043468


---
## Comparing metrics across forecasters

Since all `RunResult.metrics` share the same schema, we can easily compare.

In [14]:
comparison = pd.concat([
    result_recursive.metrics.assign(forecaster="ForecasterRecursive"),
    result_direct.metrics.assign(forecaster="ForecasterDirect"),
    result_stats.metrics.assign(forecaster="ForecasterStats"),
    result_foundation.metrics.assign(forecaster="ForecasterFoundation"),
    result_intervals.metrics.assign(forecaster="ForecasterRecursive + intervals"),
], ignore_index=True)

display(comparison[["forecaster", "series", "MAE", "MSE", "MASE"]])

,forecaster,series,MAE,MSE,MASE
0,ForecasterRecursive,sales,0.859057,1.118199,0.300112
1,ForecasterDirect,sales,0.811934,0.916786,0.283649
2,ForecasterStats,sales,1.809751,4.291044,0.632236
3,ForecasterFoundation,sales,1.758956,3.746861,0.614491
4,ForecasterRecursive + intervals,sales,1.144499,1.967030,0.399830


---
## Inspecting the generated code

Every `RunResult` includes the full standalone script. You can copy-paste it and run independently.

In [15]:
# Show code for the multi-series forecaster
print(result_multi.code)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from sklearn.linear_model import Ridge
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'revenue',
    freq      = 'D',
)

# Train/test split
end_train = '2022-06-09'  # 80% of data, adjust to change the split point
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
series_dict_test  = {k: v.loc[v.index > end_train] for k, v